# Sprint 3 : Architecture du modèle COFI (Seq2Seq LSTM)

**Objectif** : construire le modèle encodeur-décodeur LSTM, sans attention dans un premier temps, et vérifier qu'il accepte nos séquences et produit une sortie de la bonne forme, même sans entraînement.

Ce notebook est indépendant du sprint 2 : tant que le vocabulaire et la longueur maximale ne sont pas finalisés, on travaille avec des **valeurs provisoires** (section 1) et des **données aléatoires** (section 4). Quand le sprint 2 sera terminé, il suffira de remplacer les valeurs de la section 1.

## 0. Les notions à comprendre

- **RNN** : un réseau qui lit une séquence élément par élément en gardant une mémoire de ce qu'il a lu.
- **LSTM** : un RNN dont la mémoire ne s'efface pas sur les longues séquences, grâce à des portes (oubli, entrée, sortie) et à une mémoire longue `c`. Un RNN simple oublie le début d'une phrase longue.
- **Encodeur** : lit la phrase fautive et la résume dans deux vecteurs d'états (`h` et `c`).
- **Décodeur** : part de ce résumé et écrit la phrase corrigée, un token à la fois.
- **Embedding** : transforme l'identifiant d'un token en un petit vecteur de nombres, appris pendant l'entraînement.
- **Teacher forcing** : pendant l'entraînement, le décodeur reçoit en entrée la vraie phrase corrigée décalée d'un cran, pour apprendre à prédire le token suivant.

## 1. Paramètres (à remplacer par ceux du sprint 2)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --- Valeurs PROVISOIRES : à remplacer par celles du sprint 2 (étapes 2.8, 2.11, 2.12) ---
SRC_VOCAB_SIZE = 20000   # taille du vocabulaire des phrases incorrectes (tokens spéciaux inclus)
TGT_VOCAB_SIZE = 20000   # taille du vocabulaire des phrases correctes (tokens spéciaux inclus)
MAX_LEN_SRC = 40         # longueur maximale d'une phrase incorrecte
MAX_LEN_TGT = 42         # longueur maximale d'une phrase correcte (avec <SOS> et <EOS>)

# Identifiants des tokens spéciaux : à aligner sur le vocabulaire du collègue
PAD_ID, UNK_ID, SOS_ID, EOS_ID = 0, 1, 2, 3

# --- Hyperparamètres de l'architecture (nos choix) ---
EMB_DIM = 128            # taille du vecteur d'embedding
UNITS = 256              # nombre d'unités des LSTM (taille de la mémoire)

keras.utils.set_random_seed(42)   # reproductibilité : mêmes poids aléatoires à chaque exécution
print("TensorFlow", tf.__version__)

**Point à confirmer avec le collègue** : la valeur de `PAD_ID` doit être **0**. Le modèle utilise `mask_zero=True`, donc l'identifiant 0 est ignoré (il sert au padding). Si son vocabulaire donne un autre identifiant à `<PAD>`, il faut le lui faire changer.

## 2. Construction du modèle d'entraînement

Le modèle a **deux entrées** et **une sortie** :

| Entrée / sortie | Contenu | Forme |
|---|---|---|
| `encoder_input` | phrase incorrecte (identifiants, avec padding) | (lot, MAX_LEN_SRC) |
| `decoder_input` | phrase correcte décalée : `<SOS> t1 t2 ...` | (lot, MAX_LEN_TGT) |
| sortie | probabilité de chaque token du vocabulaire à chaque position | (lot, MAX_LEN_TGT, TGT_VOCAB_SIZE) |

In [ ]:
# ---------- ENCODEUR : lit la phrase fautive et la résume ----------
encoder_inputs = keras.Input(shape=(None,), name="encoder_input")

# Embedding : identifiant -> vecteur de EMB_DIM nombres appris.
# mask_zero=True : les positions de padding (id 0) sont ignorées par la suite du réseau.
encoder_emb = layers.Embedding(SRC_VOCAB_SIZE, EMB_DIM, mask_zero=True,
                               name="encoder_embedding")(encoder_inputs)

# LSTM de l'encodeur : on ne garde PAS la séquence de sorties, seulement les états finaux
# h (mémoire courte, "état caché") et c (mémoire longue, "état de cellule").
_, state_h, state_c = layers.LSTM(UNITS, return_state=True, name="encoder_lstm")(encoder_emb)

# ---------- DÉCODEUR : écrit la correction à partir du résumé ----------
decoder_inputs = keras.Input(shape=(None,), name="decoder_input")
decoder_embedding = layers.Embedding(TGT_VOCAB_SIZE, EMB_DIM, mask_zero=True,
                                     name="decoder_embedding")
decoder_emb = decoder_embedding(decoder_inputs)

# LSTM du décodeur : il démarre avec les états (h, c) de l'encodeur = le "résumé" de la phrase.
# return_sequences=True : on veut une sortie à CHAQUE position pour prédire chaque token.
decoder_lstm = layers.LSTM(UNITS, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_seq, _, _ = decoder_lstm(decoder_emb, initial_state=[state_h, state_c])

# Couche dense + softmax : transforme chaque sortie LSTM en probabilités sur tout le vocabulaire.
decoder_dense = layers.Dense(TGT_VOCAB_SIZE, activation="softmax", name="output_dense")
decoder_outputs = decoder_dense(decoder_seq)

model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs, name="cofi_seq2seq")
model.summary()

### Pourquoi ce nombre de paramètres ?

Chaque couche contribue ainsi (avec `E = EMB_DIM`, `U = UNITS`, `V` = taille du vocabulaire) :

- **Embedding** : `V x E` (un vecteur de taille E par token).
- **LSTM** : `4 x (E + U + 1) x U` (4 portes, chacune avec ses poids d'entrée, de mémoire et son biais).
- **Dense** : `U x V + V` (poids + biais pour chaque token du vocabulaire).

Les couches **Embedding** et **Dense** pèsent le plus quand le vocabulaire est grand : c'est la raison pour laquelle on limite la taille du vocabulaire (par exemple aux 20 000 mots les plus fréquents, le reste devenant `<UNK>`). Le calcul ci-dessous le vérifie.

In [ ]:
E, U = EMB_DIM, UNITS
emb_enc  = SRC_VOCAB_SIZE * E
emb_dec  = TGT_VOCAB_SIZE * E
lstm     = 4 * (E + U + 1) * U       # même formule pour l'encodeur et le décodeur
dense    = U * TGT_VOCAB_SIZE + TGT_VOCAB_SIZE
total_calcule = emb_enc + emb_dec + 2 * lstm + dense
print("Calculé à la main :", f"{total_calcule:,}")
print("Donné par Keras   :", f"{model.count_params():,}")
assert total_calcule == model.count_params()

## 3. Vérification : les poids sont initialisés aléatoirement

Aucun modèle préentraîné n'est chargé : tous les poids sont tirés au hasard à la création. Les poids de l'embedding et du LSTM sont donc des nombres sans signification, de moyenne proche de 0.

In [ ]:
for nom in ["encoder_embedding", "encoder_lstm", "decoder_lstm", "output_dense"]:
    w = model.get_layer(nom).get_weights()[0]
    print(f"{nom:20s} forme {str(w.shape):15s} moyenne {w.mean():+.4f}  écart-type {w.std():.4f}")

## 4. Vérification : forme de la sortie, sans entraînement

On fabrique de **fausses données** de la bonne forme (identifiants aléatoires, avec du padding à la fin) et on vérifie ce que le modèle produit.

In [ ]:
BATCH = 4
rng = np.random.default_rng(0)

# Identifiants aléatoires entre 4 et V-1 (on évite les 4 tokens spéciaux), puis padding à la fin
x_enc = rng.integers(4, SRC_VOCAB_SIZE, size=(BATCH, MAX_LEN_SRC)); x_enc[:, -6:] = PAD_ID
x_dec = rng.integers(4, TGT_VOCAB_SIZE, size=(BATCH, MAX_LEN_TGT)); x_dec[:, 0] = SOS_ID; x_dec[:, -4:] = PAD_ID

sortie = model.predict([x_enc, x_dec], verbose=0)
print("Forme de la sortie :", sortie.shape)
print("Attendu            :", (BATCH, MAX_LEN_TGT, TGT_VOCAB_SIZE))
print("Somme des probabilités à une position :", sortie[0, 0].sum())   # doit valoir 1
assert sortie.shape == (BATCH, MAX_LEN_TGT, TGT_VOCAB_SIZE)

**Lecture** : à chaque position de la phrase corrigée, le modèle donne une probabilité pour chacun des `TGT_VOCAB_SIZE` tokens. La somme vaut 1 (c'est le rôle du softmax). Le token prédit est celui dont la probabilité est la plus élevée.

## 5. Compilation et perte initiale (contrôle de cohérence)

- **Perte** : `sparse_categorical_crossentropy`. Les cibles sont des identifiants entiers (pas de one-hot), ce qui évite de créer un énorme tableau `(lot, longueur, V)`.
- **Optimiseur** : Adam.

Un modèle aléatoire qui ne sait rien répartit sa probabilité à peu près uniformément sur `V` tokens, donc sa perte initiale doit valoir environ **ln(V)**. Si on trouve une valeur très différente, il y a une erreur dans les données ou dans le modèle.

In [ ]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

# Cible = phrase correcte décalée d'un cran (sans <SOS>), avec <EOS> à la fin, puis padding
y = np.zeros((BATCH, MAX_LEN_TGT), dtype="int64")
y[:, :-1] = x_dec[:, 1:]            # token suivant à chaque position
y[x_dec == PAD_ID] = PAD_ID

perte = model.evaluate([x_enc, x_dec], y, verbose=0)
print(f"Perte initiale : {perte:.3f}")
print(f"ln(V)          : {np.log(TGT_VOCAB_SIZE):.3f}")

## 6. Modèles d'inférence (pour corriger une vraie phrase, sprint 5)

À l'entraînement, le décodeur reçoit la vraie correction. À l'usage, elle n'existe pas : il faut générer **token par token** :
1. L'encodeur résume la phrase fautive en `(h, c)`.
2. On donne `<SOS>` au décodeur, qui prédit le premier token.
3. On redonne ce token au décodeur avec les états mis à jour, et ainsi de suite jusqu'à `<EOS>` ou la longueur maximale.

On réutilise les **mêmes couches** (donc les mêmes poids) dans deux petits modèles.

In [ ]:
# Modèle 1 : phrase fautive -> états (h, c)
encoder_model = keras.Model(encoder_inputs, [state_h, state_c])

# Modèle 2 : (dernier token, h, c) -> (probabilités du token suivant, nouveaux h, c)
dec_state_h = keras.Input(shape=(UNITS,), name="dec_state_h")
dec_state_c = keras.Input(shape=(UNITS,), name="dec_state_c")
dec_token   = keras.Input(shape=(None,), name="dec_token")

d_emb = decoder_embedding(dec_token)
d_seq, new_h, new_c = decoder_lstm(d_emb, initial_state=[dec_state_h, dec_state_c])
d_out = decoder_dense(d_seq)
decoder_model = keras.Model([dec_token, dec_state_h, dec_state_c], [d_out, new_h, new_c])

def corriger_ids(seq_ids, max_out=MAX_LEN_TGT):
    """Décodage glouton : retourne la liste d'identifiants de la phrase corrigée."""
    h, c = encoder_model.predict(np.array(seq_ids)[None, :], verbose=0)
    token, sortie = np.array([[SOS_ID]]), []
    for _ in range(max_out):
        probas, h, c = decoder_model.predict([token, h, c], verbose=0)
        suivant = int(np.argmax(probas[0, -1]))
        if suivant == EOS_ID:
            break
        sortie.append(suivant)
        token = np.array([[suivant]])
    return sortie

print(corriger_ids(x_enc[0])[:10])   # identifiants quelconques : le modèle n'est pas entraîné

## 7. Bilan du sprint 3 (à compléter)

| Rubrique | À compléter |
|---|---|
| Ce que nous avons réalisé | Modèle Seq2Seq LSTM construit, forme de sortie vérifiée, modèles d'inférence prêts |
| Ce que nous avons appris | |
| Ce qui reste à faire | Brancher les vrais vocabulaires et séquences du sprint 2, puis entraîner (sprint 4). L'attention reste optionnelle. |
| Prochaine étape | Sprint 4 : entraînement |

### Questions possibles du professeur

1. **Pourquoi un LSTM plutôt qu'un RNN simple ?** La mémoire d'un RNN simple s'efface sur les longues séquences. Le LSTM garde une mémoire longue `c`, contrôlée par des portes.
2. **Que fait l'encodeur, que fait le décodeur ?** L'encodeur résume la phrase fautive en `(h, c)`. Le décodeur écrit la correction token par token à partir de ce résumé.
3. **Combien de paramètres, et pourquoi ?** Voir la section 2 : la plupart viennent des couches Embedding et Dense, proportionnelles à la taille du vocabulaire.
4. **Pourquoi `mask_zero=True` ?** Pour que le padding ne perturbe ni l'apprentissage ni le calcul de la perte.
5. **Limite de cette architecture ?** Tout le sens de la phrase doit tenir dans deux vecteurs de taille fixe. L'attention corrigerait cela.